In [152]:
import pypsa
import pandas as pd
import numpy as np

Parsing the datasets

In [153]:
prices_csv = pd.read_csv("entsoe_datasets/energy_prices_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
prices_csv = prices_csv[prices_csv["Sequence"] == "Sequence 1"]
price_series = []
for price in prices_csv.values:
    price_series.append(price[1])

pv_generation_csv = pd.read_csv("entsoe_datasets/pv_generation_15_07_2025.csv",
                            parse_dates=["time"],
                            index_col=["time"],
                            usecols=lambda col: "time" in col or "Day-ahead" in col,
                            date_format="%Y-%m-%d %H:%M:%S")
normalized_pv = pv_generation_csv.values #/ 32
normalized_pv = normalized_pv / normalized_pv.max()
# (pv_generation_csv.values).max()
pv_series = []
for pv in normalized_pv:
    pv_series.append(pv[0]) # kw

load_csv = pd.read_csv("entsoe_datasets/load_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
normalized_load = load_csv.values 
normalized_load = normalized_load / normalized_load.max()
load_series = []
for load in normalized_load:
    load_series.append(load[0]) # kw


Create the network and define the constant variables

In [154]:
network = pypsa.Network()
start_time = "2025-07-15 00:00:00"
timestamps = pd.date_range(start_time, periods=24, freq="h")
network.set_snapshots(timestamps)
scaling_factor = 0.05

Define the buses

In [155]:
for i in range(1, 34):
    network.add(
        "Bus",
        f"Bus_{i}",
        v_nom=12.66
    )

Define carrier

In [156]:
network.add("Carrier", "AC")
network.add("Carrier", "solar")
network.add("Carrier", "battery")
network.add("Carrier", "gas")

Index(['gas'], dtype='str')

Add the line data

In [157]:
# (from, to, r, x, s_nom)
line_data = [
    (1, 2, 0.0922, 0.047, 260),         (2, 3, 0.493, 0.2511, 230),     (3, 4, 0.366, 0.1864, 160), 
    (4, 5, 0.3811, 0.1941, 150),        (5, 6, 0.819, 0.707, 145),      (6, 7, 0.1872, 0.6188, 75), 
    (7, 8, 0.7114, 0.2351, 65),         (8, 9, 1.03, 0.74, 50),         (9, 10, 1.044, 0.74, 45),   
    (10, 11, 0.1966, 0.065, 40),        (11, 12, 0.3744, 0.198, 40),    (12, 13, 1.468, 1.155, 35),
    (13, 14, 0.5416, 0.7129, 30),       (14, 15, 0.591, 0.526, 20),     (15, 16, 0.7463, 0.545, 15),
    (16, 17, 1.289, 1.721, 15),         (17, 18, 0.732, 0.574, 10),     (2, 19, 0.164, 0.1565, 25),
    (19, 20, 1.5042, 1.3554, 20),       (20, 21, 0.4095, 0.4784, 15),   (21, 22, 0.7089, 0.9373, 10),   
    (3, 23, 0.4512, 0.3083, 65),        (23, 24, 0.898, 0.7091, 60),    (24, 25, 0.896, 0.7011, 30), 
    (6, 26, 0.203, 0.1034, 65),         (26, 27, 0.2842, 0.1447, 60),   (27, 28, 1.059, 0.9337, 60), 
    (28, 29, 0.8042, 0.7006, 55),       (29, 30, 0.5075, 0.2585, 45),   (30, 31, 0.9744, 0.963, 30), 
    (31, 32, 0.3105, 0.3619, 20),       (32, 33, 0.341, 0.5302, 5),
]

for i, (from_b, to_b, r, x, s) in enumerate(line_data):
    network.add(
        "Line", 
        f"Line_{from_b}-{to_b}",
        bus0=f"Bus_{from_b}",
        bus1=f"Bus_{to_b}",
        r=r,
        x=x,
        s_nom=s,
        carrier="AC"
    )

Define the IEEE 33 power profile

In [158]:
load_data = [
    (0,0),(100, 60), (90, 40), (120, 80), (60, 30), (60, 20),
    (200, 100), (200, 100), (60, 20), (60, 20), (45, 30),
    (60, 35), (60, 35), (120, 80), (60, 10), (60, 20),
    (60, 20), (90, 40), (90, 40), (90, 40), (90, 40),
    (90, 40), (90, 50), (420, 200), (420, 200), (60, 25),
    (60, 25), (60, 20), (120, 70), (200, 600), (150, 70),
    (210, 100), (60, 40)
]

Define the PV invertors and the batteries

In [159]:
pv_sizes = [3, 4, 5, 6, 8, 10, 12, 15, 20, 25]
# batteries = [(5, 2.5), (10, 5.0), (15, 7.5), (20, 10), (40, 15)] #(energy, power)
# batteries = [5, 10, 15, 20, 40]
batteries = [5, 10.2, 13.5, 22.1, 40.5]
stateOfHealth = 0.80
total_life_cap_loss = 1.0 - stateOfHealth
infoBatt = {5: [6000, 4500], 10.2: [6000, 7000], 13.5: [4000, 8500], 22.1: [6000, 12500], 40.5: [4000, 23500]}
cycle_life = [6000, 6000, 4000, 6000, 4000]
estimatedInstalledCost = [4500, 7000, 8500, 12500, 23500]

sel_batt_cost = []
sel_batt_info = []
selected_pv = []
selected_bat = []
for i in range(33):
    if i == 0:
        selected_pv.append(0)
        selected_bat.append(0)
        estimatedInstalledCost.append(0)
        sel_batt_info.append(0)
        continue
    pv_size = load_data[i][0] * 0.05
    idx = np.searchsorted(pv_sizes, pv_size)
    pv_size_cur = pv_sizes[min(idx, len(pv_sizes) - 1)]
    selected_pv.append(pv_size_cur)
    target_energy_bat = load_data[i][0] * 0.5 * 4 * 0.05
    bat = batteries[-1]
    cost = estimatedInstalledCost[-1]
    cycle = cycle_life[-1]
    idx = -1
    for cap in batteries:
        idx += 1
        if cap >= target_energy_bat:
            bat = cap
            cost = estimatedInstalledCost[idx]
            cycle = cycle_life[idx]
            break
    selected_bat.append(bat)
    sel_batt_cost.append(cost)
    sel_batt_info.append(cycle)

print(selected_pv)

print(selected_bat)
batt_capacity = list(selected_bat)

batt_annual_cost = []
marginal_degradation_cost = []
for i in range(32):
    if i == 0:
        batt_annual_cost.append(0)
        marginal_degradation_cost.append(0)
        continue
    print(i)
    annual_cost = sel_batt_cost[i] * (0.05 * (1 + 0.05)**15) / ((1+0.05)**15-1)
    batt_annual_cost.append(annual_cost)
    degradation_cost = sel_batt_cost[i] / selected_bat[i] * sel_batt_info[i] * 1000
    marginal_degradation_cost.append(degradation_cost)
    # capacity_mwh = selected_bat[i] / 1

[0, 5, 5, 6, 3, 3, 10, 10, 3, 3, 3, 3, 3, 6, 3, 3, 3, 5, 5, 5, 5, 5, 5, 25, 25, 3, 3, 3, 6, 10, 8, 12, 3]
[0, 10.2, 10.2, 13.5, 10.2, 10.2, 22.1, 22.1, 10.2, 10.2, 5, 10.2, 10.2, 13.5, 10.2, 10.2, 10.2, 10.2, 10.2, 10.2, 10.2, 10.2, 10.2, 40.5, 40.5, 10.2, 10.2, 10.2, 13.5, 22.1, 22.1, 22.1, 10.2]
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31


Add the loads

In [160]:
for i, (p, q) in enumerate(load_data):
    temp_load_profile = []
    real_power_profile = q* scaling_factor
    temp_load_profile = np.array(load_series) * p * scaling_factor
    real_load_profile = [scaling_factor * load_val for load_val in temp_load_profile]

    # load_profile_series = pd.Series(real_load_profile, index=network.snapshots)
    network.add(
        "Load",
        f"Load_bus_{i+1}",
        bus=f"Bus_{i+1}",
        p_set=real_load_profile,
        q_set=real_power_profile,
        carrier="AC"
    )

Add the substation

In [161]:
network.add(
    "Generator",
    "Substation",
    bus="Bus_1",
    p_nom=200,
    p_min_pu=-100,
    carrier="gas",
    marginal_cost=price_series,
)

Index(['Substation'], dtype='str')

Add the PV + batteries

In [162]:
for i in range(2, 34):

    real_pv = [selected_pv[i-1] * pv_prod for pv_prod in pv_series]


    network.add(
        "Generator",
        f"PV_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=real_pv,
        p_max_pu=pv_series,
        carrier="solar",
        marginal_cost=0,
    )
    network.add(
        "StorageUnit",
        f"Battery_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=(selected_bat[i-2]), # Nominal capacity in kW
        #p_min_pu=0.10,
        #p_max_pu=0.95,
        max_hours=4, # Energy capacity is p_nom * max_hours = 800 kWh
        carrier="battery",
        efficiency_store=0.9,
        efficiency_dispatch=0.9,
        standing_loss=0.01, # 1% loss per hour,
        #capital_cost=batt_annual_cost[i-2],
        #marginal_cost=marginal_degradation_cost[i-2]
    )

In [163]:
network.optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.05s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 3864 primals, 9288 duals
Objective: -3.10e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.
c:\Users\Raluq\anaconda3\Lib\site-packages\pypsa\network\power_flow.py:1064: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  self.p_branch_shift = np.multiply(-b, phase_shift, where=b != np.inf)


('ok', 'optimal')

In [164]:
print(f"cost: {network.objective:.2f}")

cost: -31032.62


In [165]:
network.storage_units_t.p

StorageUnit,Battery_bus_2,Battery_bus_3,Battery_bus_4,Battery_bus_5,Battery_bus_6,Battery_bus_7,Battery_bus_8,Battery_bus_9,Battery_bus_10,Battery_bus_11,...,Battery_bus_24,Battery_bus_25,Battery_bus_26,Battery_bus_27,Battery_bus_28,Battery_bus_29,Battery_bus_30,Battery_bus_31,Battery_bus_32,Battery_bus_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-07-15 00:00:00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 01:00:00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 02:00:00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 03:00:00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 04:00:00,0.0,-10.200000,-10.200000,-13.500000,-10.200000,-10.200000,-22.100000,-12.607829,-10.200000,-10.200000,...,-10.200000,-29.285618,0.000000,-10.20000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 05:00:00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 06:00:00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 07:00:00,0.0,8.016610,8.016610,10.610220,8.016610,8.016610,17.369322,9.909025,8.016610,8.016610,...,8.016610,23.016803,0.000000,8.01661,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2025-07-15 08:00:00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
